In [0]:
from pyspark.sql.functions import col, round, when, lit

# 1. Read cleaned data from Silver Managed Tables
df_pop = spark.read.table("silver_ibge_population_cleansed")
df_crime = spark.read.table("silver_sinesp_crime_data_cleansed")

# 2. Perform the JOIN between Crimes and Demographics using the clean municipality name and state
# Note: We join on municipality_name_clean to overcome differences in accents or spacing.
df_joined = df_crime.join(
    df_pop,
    df_crime["municipality_name_clean"] == df_pop["municipality_name_clean"],
    "inner"
)

# 3. Calculate the Crime Rate per 100,000 inhabitants
# Formula: (Occurrences / Estimated Population) * 100,000
df_gold_model = df_joined.withColumn(
    "crime_rate_per_100k",
    when(
        (col("population_estimated").isNotNull()) & (col("population_estimated") > 0),
        round((col("occurrences_count") / col("population_estimated")) * 100000, 2)
    ).otherwise(lit(0.0))
)

# 4. Select and organize the final analytical schema for Business Consumption
df_gold_final = df_gold_model.select(
    df_crime["state_code"],
    df_crime["municipality_name_original"].alias("municipality_name"),
    df_crime["crime_type"],
    df_crime["reference_date"],
    df_crime["occurrences_count"],
    df_pop["population_estimated"],
    col("crime_rate_per_100k")
)

# 5. Save to the Gold Layer as a Managed Delta Table
table_name_gold = "gold_municipality_crime_rate"

df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name_gold)

print(f"Gold model successfully created! {df_gold_final.count()} records processed and materialized.")
display(df_gold_final.orderBy(col("crime_rate_per_100k").desc()))

Gold model successfully created! 526512 records processed and materialized.


state_code,municipality_name,crime_type,reference_date,occurrences_count,population_estimated,crime_rate_per_100k
PA,BELÉM,Mandado de prisão cumprido,2026-03-01,153,4819,3174.93
AC,RIO BRANCO,Mandado de prisão cumprido,2026-07-01,135,4440,3040.54
SP,SANTO ANDRÉ,Mandado de prisão cumprido,2026-07-01,82,2708,3028.06
AC,RIO BRANCO,Mandado de prisão cumprido,2026-06-01,132,4440,2972.97
SP,SANTO ANDRÉ,Mandado de prisão cumprido,2026-06-01,74,2708,2732.64
PA,BELÉM,Mandado de prisão cumprido,2026-05-01,129,4819,2676.9
PA,BELÉM,Mandado de prisão cumprido,2026-07-01,129,4819,2676.9
SP,SANTO ANDRÉ,Mandado de prisão cumprido,2026-05-01,71,2708,2621.86
MS,CAMPO GRANDE,Mandado de prisão cumprido,2026-03-01,205,8285,2474.35
SP,SANTO ANDRÉ,Mandado de prisão cumprido,2026-03-01,67,2708,2474.15
